# Solar filament segmentation: from the test images to the submission

This notebook reproduces our competition submission with the files of this repository only:
the bundled weights in `weights/`, the settings in `pipeline_config.json` and the `filseg` package.
No training and no network access are needed. It runs on a GPU, or on a CPU (slower).

| Step | What happens |
|---|---|
| 1 | settings and paths |
| 2 | the weight files are checked against their SHA-256 checksums |
| 3 | the networks are loaded: an ensemble of 20 U-Nets (five with a ResNet34 encoder trained on soft (annotator-averaged) targets, five with an EfficientNetV2-S encoder, five with an EfficientNetV2-S encoder trained on soft (annotator-averaged) targets, five with an EfficientNetV2-S encoder; one per cross-validation fold each) |
| 4 | one image is followed through every step of the pipeline, with figures |
| 5 | all test images are predicted and `submission.csv` is written |
| 6 | the file format is checked (columns, RLE masks, no overlaps) |
| 7 | a few predictions are shown, and the runtime is summarised |

Headless use: `FILSEG_TEST_DIR=/path/to/test_images jupyter nbconvert --to notebook
--execute notebooks/pipeline.ipynb`.
See the README for how each step was chosen and for the training code.

## 1. Settings

**What:** locate the repository, the test images and the output file, and read the inference
settings (`pipeline_config.json`, the default P3; `FILSEG_CONFIG=P`
selects `configs/P.json`, the alternate).

**Why:** every choice that affects the predictions (weights, test-time augmentation, post-processing
parameters) lives in that one file, so the code below contains no hidden constants.
Environment variables override the defaults: `FILSEG_TEST_DIR`, `FILSEG_OUTPUT_CSV`,
`FILSEG_DEVICE` (`cuda`/`cpu`), `FILSEG_LIMIT` (first *N* images only) and `FILSEG_THREADS`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "pipeline_config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

TEST_IMAGE_DIR = Path(os.environ.get("FILSEG_TEST_DIR", ROOT / "test_images"))
OUTPUT_CSV = Path(os.environ.get("FILSEG_OUTPUT_CSV", ROOT / "submission.csv"))
LIMIT = int(os.environ.get("FILSEG_LIMIT") or 0)  # 0 = all images
DEVICE = os.environ.get("FILSEG_DEVICE") or None  # None = CUDA if available, else CPU
THREADS = int(os.environ.get("FILSEG_THREADS") or 1)  # 1 = verified release setting

CONFIG_NAME = os.environ.get("FILSEG_CONFIG") or None  # None = default (pipeline_config.json)
if CONFIG_NAME:
    _config_path = ROOT / "configs" / f"{CONFIG_NAME}.json"
    if not _config_path.exists():
        raise FileNotFoundError(f"unknown FILSEG_CONFIG {CONFIG_NAME!r}; see configs/ for the bundled configurations")
else:
    _config_path = ROOT / "pipeline_config.json"
config = json.loads(_config_path.read_text())
print("repository:", ROOT)
print("configuration:", _config_path)
print("test images:", TEST_IMAGE_DIR)
print(json.dumps(config, indent=2))

## 2. Check the weight files

**What:** recompute the SHA-256 checksum of every weight file and compare it with `weights/manifest.json`.

**Why:** a truncated download or a Git LFS pointer file would otherwise fail later with a confusing error,
or silently change the predictions.

In [ ]:
import hashlib

manifest = json.loads((ROOT / "weights" / "manifest.json").read_text())
if "files" in manifest:
    _selected = CONFIG_NAME or manifest.get("default")
    entries = [e for e in manifest["files"] if _selected in (e.get("used_by") or [])]
else:
    entries = manifest["weights"] + manifest["refiner_weights"]
    entries += [manifest["tree_model"]] if "tree_model" in manifest else []
for entry in entries:
    path = ROOT / "weights" / entry["path"]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != entry["sha256"]:
        raise RuntimeError(f"{path}: checksum mismatch - re-download the file (Git LFS pointer or truncated copy?)")
print(f"{len(entries)} weight files verified for {_selected if 'files' in manifest else 'this release'}")

## 3. Load the pipeline

**What:** load an ensemble of 20 U-Nets (five with a ResNet34 encoder trained on soft (annotator-averaged) targets, five with an EfficientNetV2-S encoder, five with an EfficientNetV2-S encoder trained on soft (annotator-averaged) targets, five with an EfficientNetV2-S encoder; one per cross-validation fold each); every network is rebuilt from the architecture recorded in its
checkpoint and receives its weights.

**Why:** one model per cross-validation fold was trained on the other four folds; averaging the 20
models at test time uses all training images and reduces the variance of the prediction. On a GPU the
networks run with FP16 autocast, as during validation; on a CPU they run in FP32, which changes the
probabilities only at the level of rounding.

In [ ]:
import random
import resource

import cv2
import numpy as np
import torch

from filseg.infer import FilamentPipeline, PipelineConfig

if THREADS < 1:
    raise ValueError('FILSEG_THREADS must be at least 1')
torch.set_num_threads(THREADS)
cv2.setNumThreads(THREADS)
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
pipeline = FilamentPipeline(PipelineConfig.from_json(_config_path), ROOT / "weights", device=DEVICE)
pipeline.warm_up()  # CUDA start-up is excluded from the timings below
if pipeline.device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
print(f'Python {sys.version.split()[0]}, torch {torch.__version__}, CUDA {torch.version.cuda}, '
      f'cuDNN {torch.backends.cudnn.version()}, threads {THREADS}')
encoders = sorted({str(args.get("encoder")) for args in pipeline.model_args})
print(
    f"{len(pipeline.models)} models ({', '.join(encoders)}) on {pipeline.device}, precision {pipeline.precision}; "
    f"{pipeline.n_views} test-time views -> {len(pipeline.models) * pipeline.n_views} forward passes per image"
)

stems = sorted(p.stem for p in TEST_IMAGE_DIR.glob("*.jpeg"))
if not stems:
    raise FileNotFoundError(f"no .jpeg images in {TEST_IMAGE_DIR}; set FILSEG_TEST_DIR")
if LIMIT:
    stems = stems[:LIMIT]
print(f"{len(stems)} test images")

## 4. One image, step by step

**What:** run the pipeline on the first test image and keep every intermediate result.

1. **Input.** The 2048×2048 H-alpha image; the networks see the central 1888×1888 crop (dashed box), which
   contains the whole solar disk, at native resolution, because many filaments are only a few pixels wide.
2. **Probability.** The 20 models × 8 views are averaged into one probability per pixel (160 forward passes).
3. **Threshold.** The tree thresholds the probability at 15 levels; every component of at least 50 px is a candidate node.
4. **Closing.** Disabled in this configuration (`tree_close_radius` = 0), so the forest is exactly nested.
5. **Mean-probability gate.** A LightGBM model scores every tree node and an exact dynamic program keeps the disjoint set with the highest total score (tau = 0.28); faint, diffuse blobs score too low to be kept, while real filaments have a confident core. Kept candidates are green, removed ones red.
6. **Instances.** Each remaining component is one filament; every pixel belongs to at most one instance.

**Why:** the figure makes each decision of the post-processing visible on real data. Panels 2-6 show the
orange window of panel 1, centred on the largest predicted filament.

In [ ]:
from filseg.postprocess import trace_postprocess
from filseg.preprocess import read_image

example_stem = stems[0]
example_image = read_image(TEST_IMAGE_DIR / f"{example_stem}.jpeg")
example = pipeline.predict(example_image)
if config.get("postprocess_method") == "component_tree":
    # Tree mode: panels 3-5 show the lowest tree level and the selected nodes.
    from types import SimpleNamespace

    import numpy as _np

    stage1 = pipeline.labels_from_probability(example_image, example.probability)
    _binary = (example.probability > 0.05).astype(_np.uint8)
    trace = SimpleNamespace(
        binary=_binary, closed=_binary, candidates=stage1,
        areas=_np.bincount(stage1.ravel())[1:], kept=_np.ones(int(stage1.max()), bool), labels=stage1,
    )
    print(f"{example_stem}: component tree selected {int(stage1.max())} stage-1 instances "
          f"(tree_tau={config.get('tree_tau')}), {int(example.labels.max())} after the refiner")
else:
    trace = trace_postprocess(example.probability, config["postprocess"])
    too_small = trace.areas < config["postprocess"].get("min_area", 0)
    print(
        f"{example_stem}: {len(trace.areas)} candidate components -> {int(too_small.sum())} too small, "
        f"{int((~too_small & ~trace.kept).sum())} below the mean-probability gate, {int(trace.kept.sum())} instances"
    )
print(f"time: {example.infer_s:.1f} s networks, {example.postprocess_s:.2f} s post-processing")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

from filseg.preprocess import CROP_OFFSET, CROP_SIZE


def zoom_window(labels: np.ndarray, size: int = 512) -> tuple[slice, slice]:
    """Square window around the largest instance (the image centre if there is none)."""
    cy = cx = labels.shape[0] // 2
    if labels.max() > 0:
        ys, xs = np.nonzero(labels == np.argmax(np.bincount(labels.ravel())[1:]) + 1)
        cy, cx = int(ys.mean()), int(xs.mean())
    y0 = int(np.clip(cy - size // 2, 0, labels.shape[0] - size))
    x0 = int(np.clip(cx - size // 2, 0, labels.shape[1] - size))
    return slice(y0, y0 + size), slice(x0, x0 + size)


def colour_layers(shape: tuple, layers: list) -> np.ndarray:
    """RGB image from (mask, colour) pairs drawn in order."""
    rgb = np.zeros((*shape, 3))
    for mask, colour in layers:
        rgb[mask] = colour
    return rgb


def show_instances(ax, image: np.ndarray, labels: np.ndarray, title: str) -> None:
    """Grayscale image with one random colour per instance."""
    n = max(int(labels.max()), 1)
    colours = ListedColormap(np.random.default_rng(0).uniform(0.2, 1.0, size=(n, 3)))
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.imshow(np.ma.masked_equal(labels, 0), cmap=colours, vmin=1, vmax=n, alpha=0.65, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")


pp = config["postprocess"]
win = zoom_window(trace.labels)
kept_ids = np.flatnonzero(trace.kept) + 1
fig, axes = plt.subplots(2, 3, figsize=(15, 10.5))
ax = axes[0, 0]
ax.imshow(example_image, cmap="gray", vmin=0, vmax=255)
ax.add_patch(Rectangle((CROP_OFFSET, CROP_OFFSET), CROP_SIZE, CROP_SIZE, fill=False, ec="deepskyblue", ls="--", lw=1.5))
ax.add_patch(Rectangle((win[1].start, win[0].start), 512, 512, fill=False, ec="orange", lw=2))
ax.set_title(f"1. input {example_stem}")
ax.axis("off")
ax = axes[0, 1]
shown = ax.imshow(example.probability[win], cmap="magma", vmin=0, vmax=1)
fig.colorbar(shown, ax=ax, fraction=0.046, pad=0.02)
ax.set_title("2. ensemble probability")
ax.axis("off")
axes[0, 2].imshow(trace.binary[win], cmap="gray")
if config.get("postprocess_method") == "component_tree":
    axes[0, 2].set_title("3. lowest tree level: p > 0.05")
else:
    axes[0, 2].set_title(f"3. threshold: p > {pp['threshold']}")
axes[0, 2].axis("off")
added = (trace.closed > 0) & (trace.binary == 0)
axes[1, 0].imshow(colour_layers(added[win].shape, [(trace.binary[win] > 0, (1, 1, 1)), (added[win], (0, 0.8, 1))]))
if config.get("postprocess_method") == "component_tree":
    axes[1, 0].set_title("4. closing disabled (exactly nested forest)")
else:
    axes[1, 0].set_title(f"4. closing, radius {pp.get('close_radius', 0)} (added pixels in cyan)")
axes[1, 0].axis("off")
kept = np.isin(trace.candidates, kept_ids) & (trace.candidates > 0)
removed = (trace.candidates > 0) & ~kept
axes[1, 1].imshow(colour_layers(kept[win].shape, [(kept[win], (0.2, 0.8, 0.3)), (removed[win], (0.9, 0.2, 0.2))]))
if config.get("postprocess_method") == "component_tree":
    axes[1, 1].set_title(f"5. tree selection, tau = {config.get('tree_tau')} (kept in green)")
else:
    axes[1, 1].set_title(f"5. gate: area >= {pp.get('min_area', 0)} px, mean p >= {pp.get('min_mean', '-')}")
axes[1, 1].axis("off")
show_instances(axes[1, 2], example_image[win], example.labels[win], f"6. {int(example.labels.max())} instances")
plt.tight_layout()
plt.show()

## 5. Predict all test images and write the submission

**What:** the same pipeline for every test image; the instances of each image become rows
`filament_id = <stem>_<k>` (largest first) with the COCO run-length encoding of the mask.

**Why:** this produces the submission file. The time spent in the networks and in the post-processing is
recorded per image in `submission.csv.timing.json` (the image of step 4 is not computed twice).

In [ ]:
from filseg.submission import labels_to_rows, write_submission

rows, timings, gallery = [], [], []
loop_start = time.perf_counter()
for i, stem in enumerate(stems, 1):
    if stem == example_stem:
        image, result = example_image, example
    else:
        image = read_image(TEST_IMAGE_DIR / f"{stem}.jpeg")
        result = pipeline.predict(image)
    rows += labels_to_rows(stem, result.labels)
    timings.append(
        {
            "stem": stem, "infer_s": result.infer_s, "postprocess_s": result.postprocess_s,
            "n_instances": int(result.labels.max()),
        }
    )
    if len(gallery) < 4:
        gallery.append((stem, image, result.labels))
    if i % 20 == 0 or i == len(stems):
        print(f"{i}/{len(stems)} images, {len(rows)} instances, {time.perf_counter() - loop_start:.0f} s")
wall_s = time.perf_counter() - loop_start + example.infer_s + example.postprocess_s

write_submission(rows, OUTPUT_CSV)
per_image_s = [t["infer_s"] + t["postprocess_s"] for t in timings]
Path(f"{OUTPUT_CSV}.timing.json").write_text(
    json.dumps(
        {
            "device": str(pipeline.device),
            "precision": pipeline.precision,
            "n_images": len(stems),
            "wall_s": wall_s,
            "mean_s_per_image": sum(per_image_s) / len(per_image_s),
            "per_image": timings,
            "median_s_per_image": float(np.median(per_image_s)),
            "peak_gpu_bytes": int(torch.cuda.max_memory_allocated()) if pipeline.device.type == "cuda" else 0,
            "peak_rss_bytes": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss) * (1 if sys.platform == "darwin" else 1024),
            "csv_sha256": hashlib.sha256(OUTPUT_CSV.read_bytes()).hexdigest(),
            "n_instances": len(rows),
        },
        indent=2,
    )
)
print(f"wrote {OUTPUT_CSV}: {len(rows)} instances for {len(stems)} images")

## 6. Check the submission file

**What:** re-read the CSV and check the two columns, unique `<stem>_<k>` identifiers, that every RLE string
decodes to a non-empty 2048×2048 mask, that the instances of one image never overlap, and that every stem is a
test image.

**Why:** the competition rejects overlapping instances; checking the written file, rather than the arrays in
memory, also covers the encoding.

In [ ]:
from filseg.submission import validate_submission

print(json.dumps(validate_submission(OUTPUT_CSV, expected_stems=stems), indent=2))
print("CSV SHA-256:", hashlib.sha256(OUTPUT_CSV.read_bytes()).hexdigest())

## 7. More predictions and the runtime

**What:** the predicted instances of the first test images (one colour per instance), then the time per image.

**Why:** a quick visual check that the masks follow the dark, elongated filaments on the disk. The network
part dominates the runtime; it grows linearly with the number of models × test-time views.

In [ ]:
fig, axes = plt.subplots(1, len(gallery), figsize=(4.5 * len(gallery), 4.8), squeeze=False)
for ax, (stem, image, labels) in zip(axes[0], gallery, strict=True):
    show_instances(ax, image, labels, f"{stem}: {int(labels.max())} instances")
plt.tight_layout()
plt.show()

In [ ]:
mean_infer = sum(t["infer_s"] for t in timings) / len(timings)
mean_pp = sum(t["postprocess_s"] for t in timings) / len(timings)
print(f"device {pipeline.device}, precision {pipeline.precision}, {len(timings)} images")
print(f"per image: {mean_infer:.2f} s networks + {mean_pp:.3f} s post-processing")
print(f"total {wall_s:.0f} s; all 180 test images would take about {180 * (mean_infer + mean_pp) / 60:.0f} min here")

## 8. How the bundled weights were trained (outline, not executed here)

The full pipeline is: **data preparation → training → inference →
post-processing → submission**. This notebook executes the last three;
the first two are reproduced with the repository code as follows (see the
README, "Reproduce the training", for the exact commands; nothing in this
section is run here and no training is needed to reproduce the submission).

1. **Data preparation.** `python -m filseg.prepare` crops every training
   image to its central 1888×1888 square and stores one mask per annotator
   (`prepared/`); `folds.csv` splits the 707 images into 5 folds by image.
2. **Segmentation training (S, V, V_soft and V60k).** Per fold, stage 1
   trains on 1024×1024 crops (30,000 steps; 60,000 for V60k) and stage 2
   continues on the full 1888×1888 view for 2,000 steps
   (`filseg/train.py`: AdamW, cosine decay, BCE + 0.5 × Dice, EMA
   weights). S is ResNet34 with soft (annotator-mean) targets; V and
   V60k are EfficientNetV2-S with individual targets; V_soft is
   EfficientNetV2-S with soft targets. The default P3
   averages all four families with staged quantization; the alternate
   P averages S and V.
3. **Tree fitting.** The component tree is built on the out-of-fold
   probability maps, every node is labelled with its exact IoU
   contribution against the training annotations, and a LightGBM
   regressor learns those contributions; the released models are
   refitted on all 707 images (bagged 7+15 leaves for P3,
   a single 15-leaf model for P).
4. **Refiner training.** Each stage-1 instance becomes a 512×160
   rotation-normalised crop whose target is the matched annotators' mean
   mask; a 4-channel ResNet34 U-Net is trained per fold (2,500 steps,
   batch 24) and the cross-fit models are averaged at test time
   (10 models for P3, 5 for
   P).